In [ ]:
!pip install -q --upgrade pip

!pip install -q \
  unsloth \
  transformers \
  trl \
  peft \
  accelerate \
  bitsandbytes \
  sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 12.9.4 which is incompatible.


In [ ]:
!pip install -q --upgrade pip

!pip install -q \
  unsloth \
  transformers \
  trl \
  peft \
  accelerate \
  bitsandbytes \
  sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


CHANGE ADAPTER PATHS ACCORDINGLY

## Update File Paths

Before running inference, update the following paths according to the model you would like to evaluate.

```python
TEST_CSV = "/path/to/test_dataset.csv"
OUTPUT_CSV = "/path/to/output_predictions.csv"
ADAPTER_PATH = "/path/to/model_adapter"
```

- **`TEST_CSV`**: Path to the evaluation dataset.
- **`OUTPUT_CSV`**: Path where the generated predictions will be saved.
- **`ADAPTER_PATH`**: Path to the trained LoRA adapter that will be loaded for inference.

---

## Default Test Dataset

```python
TEST_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test_subset.csv"
```

---

## Pre-trained Adapter Paths

### SFT Adapter

```python
ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/sft_adapter"
```

Save predictions to:

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter_predictions.csv"
```

---

### GRPO Model 1 — Exact Ranking Reward

```python
ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/exact_ranking_reward_adapter"
```

Example output locations:

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/exact_ranking_reward_run1.csv"
```

or

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/exact_ranking_reward_run2.csv"
```

---

### GRPO Model 2 — Relative Ranking Reward

```python
ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/relative_ranking_reward_adapter"
```

Example output locations:

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/relative_ranking_reward_run1.csv"
```

or

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/relative_ranking_reward_run2.csv"
```

---

### GRPO Model 3 — Composite Ranking Reward

```python
ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/composite_ranking_reward_adapter"
```

Example output locations:

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/composite_ranking_reward_run1.csv"
```

or

```python
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/composite_ranking_reward_run2.csv"
```

---

## Running Inference on Your Own Adapter

If you have trained a new SFT or GRPO model, simply replace the adapter and output paths with your own.

Example:

```python
ADAPTER_PATH = "/content/drive/MyDrive/my_project/my_adapter"

OUTPUT_CSV = "/content/drive/MyDrive/my_project/my_predictions.csv"
```

The notebook will automatically load the specified adapter, run inference on the test dataset, and save the generated predictions to the specified output location.

In [ ]:
#explicitly re run the inference code for multiple inferences. This code will only provide inference for 1 run

In [ ]:
TEST_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test_subset.csv"
OUTPUT_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter_predictions.csv"
ADAPTER_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/sft_adapter"

In [ ]:
import re
import torch
import pandas as pd
from tqdm import tqdm

from unsloth import FastLanguageModel
from peft import PeftModel


BASE_MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 2048
MAX_NEW_TOKENS = 32

VALID_RANKINGS = [
    "R1>R2>R3",
    "R1>R3>R2",
    "R2>R1>R3",
    "R2>R3>R1",
    "R3>R1>R2",
    "R3>R2>R1",
]

RANKING_RE = re.compile(r"R[123]\s*>\s*R[123]\s*>\s*R[123]")


def normalize_ranking(text):
    if text is None:
        return None

    match = RANKING_RE.search(str(text).strip())
    if not match:
        return None

    ranking = match.group(0).replace(" ", "")
    return ranking if ranking in VALID_RANKINGS else None


def make_prompt(row):
    return f"""You are an expert cybersecurity answer evaluator.

You will be given a cybersecurity question and three candidate answers.

Your task is to rank the three answers from best to worst based on:
1. Technical correctness
2. Completeness
3. Relevance to the question
4. Clarity and precision
5. Lack of hallucination or misleading information
6. Keywords match

Assign each rubric a score and then sum all 6 rubric scores to find total score for a response and then rank the responses.

Important:
- R1, R2, and R3 are all candidate answers.
- Do not assume the reference answer is always best.
- Judge only based on answer quality.
- Output ONLY the ranking.
- The output format must be exactly like one of these:
R1>R2>R3
R1>R3>R2
R2>R1>R3
R2>R3>R1
R3>R1>R2
R3>R2>R1

Question:
{row["question"]}

R1:
{row["answer"]}

R2:
{row["candidate_llama_3_2_1b_instruct"]}

R3:
{row["candidate_qwen3_32b"]}

Ranking:"""


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
    is_trainable=False,
)

FastLanguageModel.for_inference(model)

print("Loaded base model:", BASE_MODEL_NAME)
print("Loaded adapter:", ADAPTER_PATH)



df = pd.read_csv(TEST_CSV)

required_cols = [
    "question",
    "answer",
    "answer_word_count",
    "dataset_name",
    "candidate_llama_3_2_1b_instruct",
    "candidate_qwen3_32b",
    "gpt_51_judge_ranking",
    "source_packet",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=required_cols).reset_index(drop=True)

df["gold_ranking"] = df["gpt_51_judge_ranking"].apply(normalize_ranking)
df = df.dropna(subset=["gold_ranking"]).reset_index(drop=True)

df["prompt"] = df.apply(make_prompt, axis=1)


def count_prompt_tokens(prompt):
    return len(
        tokenizer(
            prompt,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )


df["prompt_tokens"] = df["prompt"].apply(count_prompt_tokens)

before = len(df)
df = df[df["prompt_tokens"] <= MAX_PROMPT_LENGTH].reset_index(drop=True)
after = len(df)

print(f"Kept {after}/{before} rows after filtering")

raw_outputs = []
predictions = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Running inference"):

    inputs = tokenizer(
        row["prompt"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            repetition_penalty=1.2,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    raw_outputs.append(generated)
    predictions.append(normalize_ranking(generated))


df["raw_output"] = raw_outputs
df["prediction"] = predictions


final_cols = [
    "question",
    "answer",
    "answer_word_count",
    "dataset_name",
    "candidate_llama_3_2_1b_instruct",
    "candidate_qwen3_32b",
    "gold_ranking",
    "source_packet",
    "raw_output",
    "prediction",
]

final_df = df[final_cols]

final_df.to_csv(OUTPUT_CSV, index=False)

print("Saved predictions to:", OUTPUT_CSV)
print(final_df.head())